# Capítulo 5 · Generalización, Underfitting y Overfitting

## Diplomado en Data Engineering

### Laboratorio interactivo

En este cuaderno observaremos cómo cambia el comportamiento de un modelo cuando aumenta su complejidad.

Trabajaremos con un **dataset sintético**, diseñado especialmente para mostrar de manera clara:

- Underfitting.
- Buena generalización.
- Overfitting.
- Diferencia entre desempeño de entrenamiento y prueba.
- Trade-off entre sesgo y varianza.

## Objetivos

Al finalizar el laboratorio, el estudiante será capaz de:

- Reconocer visualmente un modelo con underfitting.
- Identificar señales de overfitting.
- Comparar el desempeño en entrenamiento y prueba.
- Analizar el efecto de aumentar la complejidad de un modelo.
- Seleccionar un modelo que generalice adecuadamente.

## 1. Importación de librerías

Utilizaremos:

- `NumPy` para generar datos.
- `Pandas` para organizar resultados.
- `Plotly` para crear gráficos interactivos.
- `Scikit-learn` para entrenar modelos polinómicos.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Generación del conjunto de datos

Generaremos una relación no lineal con ruido:

\[
y = \sin(3x) + arepsilon
\]

donde \(arepsilon\) representa variación aleatoria.

El conjunto será pequeño para que un modelo de alta complejidad pueda memorizar fácilmente los datos de entrenamiento.

In [2]:
# Generación del dataset sintético
n_observaciones = 36

X = np.sort(
    np.random.uniform(-1, 1, n_observaciones)
).reshape(-1, 1)

ruido = np.random.normal(
    loc=0,
    scale=0.25,
    size=n_observaciones
)

y = np.sin(3 * X.ravel()) + ruido

# Separación entre entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=RANDOM_STATE
)

print(f"Observaciones totales: {len(X)}")
print(f"Entrenamiento: {len(X_train)}")
print(f"Prueba: {len(X_test)}")

Observaciones totales: 36
Entrenamiento: 24
Prueba: 12


## 3. Visualización de los datos

Los puntos de entrenamiento se utilizarán para ajustar el modelo.

Los puntos de prueba se mantendrán separados y se utilizarán únicamente para evaluar la capacidad de generalización.

In [3]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_train.ravel(),
    y=y_train,
    mode="markers",
    name="Entrenamiento",
    marker=dict(size=10),
    hovertemplate=(
        "x: %{x:.3f}<br>"
        "y: %{y:.3f}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Scatter(
    x=X_test.ravel(),
    y=y_test,
    mode="markers",
    name="Prueba",
    marker=dict(size=11, symbol="diamond"),
    hovertemplate=(
        "x: %{x:.3f}<br>"
        "y: %{y:.3f}"
        "<extra></extra>"
    )
))

fig.update_layout(
    title="Datos de entrenamiento y prueba",
    xaxis_title="Variable de entrada x",
    yaxis_title="Variable objetivo y",
    height=520,
    hovermode="closest"
)

fig.show()

### Análisis

El conjunto presenta una relación no lineal y cierta dispersión causada por el ruido.

Un modelo muy simple no podrá capturar adecuadamente la forma de los datos. En cambio, un modelo excesivamente complejo puede intentar ajustarse a cada observación y terminar aprendiendo también el ruido.

## 4. Regresión polinómica interactiva

Utiliza el control deslizante para cambiar el grado del polinomio entre 1 y 15.

En cada grado se mostrarán:

- Curva estimada.
- R² de entrenamiento.
- R² de prueba.
- Diferencia entre ambos valores.
- Diagnóstico automático.

In [4]:
# Valores para dibujar una curva suave
x_curva = np.linspace(
    X.min() - 0.05,
    X.max() + 0.05,
    500
).reshape(-1, 1)

fig = go.Figure()

# Datos de entrenamiento
fig.add_trace(go.Scatter(
    x=X_train.ravel(),
    y=y_train,
    mode="markers",
    name="Entrenamiento",
    marker=dict(size=9),
    visible=True
))

# Datos de prueba
fig.add_trace(go.Scatter(
    x=X_test.ravel(),
    y=y_test,
    mode="markers",
    name="Prueba",
    marker=dict(size=10, symbol="diamond"),
    visible=True
))

resultados = []

for grado in range(1, 16):

    modelo = Pipeline([
        (
            "polinomio",
            PolynomialFeatures(
                degree=grado,
                include_bias=False
            )
        ),
        (
            "regresion",
            LinearRegression()
        )
    ])

    modelo.fit(X_train, y_train)

    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)
    pred_curva = modelo.predict(x_curva)

    r2_train = r2_score(y_train, pred_train)
    r2_test = r2_score(y_test, pred_test)
    gap = r2_train - r2_test

    # Diagnóstico basado en el comportamiento real
    if r2_train < 0.70 and r2_test < 0.70:
        diagnostico = "Underfitting"
    elif gap > 0.25:
        diagnostico = "Overfitting"
    elif gap > 0.10:
        diagnostico = "Posible sobreajuste"
    else:
        diagnostico = "Buena generalización"

    resultados.append({
        "Grado": grado,
        "R² entrenamiento": r2_train,
        "R² prueba": r2_test,
        "Diferencia": gap,
        "Diagnóstico": diagnostico
    })

    fig.add_trace(go.Scatter(
        x=x_curva.ravel(),
        y=pred_curva,
        mode="lines",
        line=dict(width=4),
        name=f"Modelo grado {grado}",
        visible=(grado == 1),
        hovertemplate=(
            f"Grado: {grado}<br>"
            "x: %{x:.3f}<br>"
            "Predicción: %{y:.3f}"
            "<extra></extra>"
        )
    ))

pasos = []

for grado in range(1, 16):

    visible = [True, True] + [False] * 15
    visible[grado + 1] = True

    r = resultados[grado - 1]

    titulo = (
        f"Regresión polinómica interactiva · Grado {grado}"
        f"<br>R² entrenamiento = {r['R² entrenamiento']:.3f}"
        f" | R² prueba = {r['R² prueba']:.3f}"
        f" | Diferencia = {r['Diferencia']:.3f}"
        f" | {r['Diagnóstico']}"
    )

    pasos.append({
        "method": "update",
        "label": str(grado),
        "args": [
            {"visible": visible},
            {"title": titulo}
        ]
    })

r_inicial = resultados[0]

fig.update_layout(
    title=(
        "Regresión polinómica interactiva · Grado 1"
        f"<br>R² entrenamiento = {r_inicial['R² entrenamiento']:.3f}"
        f" | R² prueba = {r_inicial['R² prueba']:.3f}"
        f" | Diferencia = {r_inicial['Diferencia']:.3f}"
        f" | {r_inicial['Diagnóstico']}"
    ),
    xaxis_title="Variable de entrada x",
    yaxis_title="Variable objetivo y",
    height=650,
    hovermode="closest",
    sliders=[{
        "active": 0,
        "currentvalue": {
            "prefix": "Grado del polinomio: ",
            "font": {"size": 16}
        },
        "pad": {"t": 60},
        "steps": pasos
    }]
)

fig.show()

### Guía de interpretación

#### Grado bajo

La curva es demasiado simple y no captura la estructura real de los datos.

**Resultado esperado:** underfitting.

#### Grado intermedio

La curva captura el patrón principal sin seguir cada pequeña variación.

**Resultado esperado:** buena generalización.

#### Grado alto

La curva comienza a oscilar para acercarse excesivamente a los puntos de entrenamiento.

**Resultado esperado:** overfitting.

## 5. Tabla de resultados

La siguiente tabla permite comparar todos los grados utilizados.

In [5]:
df_resultados = pd.DataFrame(resultados)

df_resultados.style.format({
    "R² entrenamiento": "{:.3f}",
    "R² prueba": "{:.3f}",
    "Diferencia": "{:.3f}"
})

,Grado,R² entrenamiento,R² prueba,Diferencia,Diagnóstico
0,1,0.614,0.487,0.127,Underfitting
1,2,0.614,0.490,0.124,Underfitting
2,3,0.932,0.915,0.017,Buena generalización
3,4,0.932,0.913,0.019,Buena generalización
4,5,0.944,0.876,0.068,Buena generalización
5,6,0.946,0.895,0.050,Buena generalización
6,7,0.950,0.765,0.186,Posible sobreajuste
7,8,0.951,0.663,0.287,Overfitting
8,9,0.951,0.461,0.489,Overfitting
9,10,0.951,-0.067,1.018,Overfitting


## 6. Evolución del rendimiento según la complejidad

El siguiente gráfico muestra cómo cambian los R² de entrenamiento y prueba al aumentar el grado del polinomio.

In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_resultados["Grado"],
    y=df_resultados["R² entrenamiento"],
    mode="lines+markers",
    name="R² entrenamiento",
    hovertemplate=(
        "Grado: %{x}<br>"
        "R² entrenamiento: %{y:.3f}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Scatter(
    x=df_resultados["Grado"],
    y=df_resultados["R² prueba"],
    mode="lines+markers",
    name="R² prueba",
    hovertemplate=(
        "Grado: %{x}<br>"
        "R² prueba: %{y:.3f}"
        "<extra></extra>"
    )
))

mejor_fila = df_resultados.loc[
    df_resultados["R² prueba"].idxmax()
]

fig.add_trace(go.Scatter(
    x=[mejor_fila["Grado"]],
    y=[mejor_fila["R² prueba"]],
    mode="markers+text",
    name="Mejor generalización",
    text=["Mejor generalización"],
    textposition="top center",
    marker=dict(size=15)
))

fig.update_layout(
    title="R² de entrenamiento y prueba según el grado del polinomio",
    xaxis_title="Grado del polinomio",
    yaxis_title="R²",
    height=540,
    hovermode="x unified"
)

fig.show()

print(
    "Mejor grado según R² de prueba:",
    int(mejor_fila["Grado"])
)

print(
    "R² de prueba:",
    round(mejor_fila["R² prueba"], 3)
)

Mejor grado según R² de prueba: 3
R² de prueba: 0.915


### Interpretación

- El R² de entrenamiento tiende a aumentar cuando crece la complejidad.
- El R² de prueba puede mejorar inicialmente.
- Después de cierto punto, aumentar la complejidad perjudica el desempeño sobre datos nuevos.
- El mejor modelo es el que obtiene el mejor equilibrio entre entrenamiento y prueba.

## 7. Diferencia entre entrenamiento y prueba

Una diferencia grande entre ambos R² es una señal de que el modelo puede estar memorizando los datos de entrenamiento.

In [7]:
fig = px.bar(
    df_resultados,
    x="Grado",
    y="Diferencia",
    hover_data=[
        "R² entrenamiento",
        "R² prueba",
        "Diagnóstico"
    ],
    title="Brecha entre R² de entrenamiento y R² de prueba"
)

fig.add_hline(
    y=0.10,
    line_dash="dash",
    annotation_text="Señal de posible sobreajuste",
    annotation_position="top left"
)

fig.add_hline(
    y=0.25,
    line_dash="dash",
    annotation_text="Señal fuerte de overfitting",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="Grado del polinomio",
    yaxis_title="R² entrenamiento − R² prueba",
    height=520
)

fig.show()

## 8. Resumen de los tres escenarios

| Escenario | Entrenamiento | Prueba | Interpretación |
|---|---|---|---|
| Underfitting | Bajo | Bajo | El modelo no aprende el patrón |
| Buena generalización | Alto | Alto | El modelo aprende y funciona con datos nuevos |
| Overfitting | Muy alto | Bajo | El modelo memoriza el entrenamiento |

## 9. Preguntas para discusión

1. ¿Por qué el R² de entrenamiento aumenta al incrementar el grado?
2. ¿Por qué un mejor resultado en entrenamiento no garantiza un mejor modelo?
3. ¿Qué grado produjo la mejor generalización?
4. ¿En qué momento comenzó a observarse overfitting?
5. ¿Qué diferencia existe entre memorizar y aprender?
6. ¿Qué podríamos hacer para reducir el overfitting?

## Conclusiones

- El objetivo del Machine Learning no es memorizar los datos.
- Un modelo demasiado simple presenta underfitting.
- Un modelo excesivamente complejo puede presentar overfitting.
- La comparación entre entrenamiento y prueba permite evaluar la generalización.
- El mejor modelo suele encontrarse en un nivel intermedio de complejidad.
- La siguiente etapa consiste en estudiar métodos más robustos de evaluación, como la validación cruzada.